# EBBA / EBBA+ 可复现验证（ICML 2024）

**论文**：Yudong Gao, Honglong Chen, Peng Sun, Zhaojie Li, Jilong Li, Shaojie Shao.
*Energy-based Backdoor Defense without Task-Specific Samples and Model Retraining*.
ICML 2024, PMLR 235. 本地文件：`paper/paper.pdf`。

这套 Notebook 的目的不是“跑一遍作者的实验”，而是把论文中**可以被独立核对**的断言逐条变成代码，
给出可复核的判定。全部数值由 `src/ebba` 计算并写入 `reports/claims/*.json`；
Notebook 只负责解释与展示，因此同样的检查可以脱离 Jupyter 用 `pytest` 运行。

## 论文主张的两件事

**EBBA（检测）**：不给任务相关样本，只拿一批任务无关图像（论文用互联网图片），
按模型伪标签每类取 N 张；把每条 logits 的最大项置 0（Eq.3），softmax 后逐标签求能量（Eq.4-6），
再对标签能量做 z-score（Eq.7）。若某个标签满足

$$E_k' - \mu > \lambda \sigma \tag{Eq.8}$$

则判定模型被植入后门，且该标签就是目标标签。

**EBBA+（样本检测 + 去除）**：对每张图施加 J 种损坏，比较损坏前后同一标签的 softmax 概率：

$$\mathrm{TE}_k^j = \frac{e^{s_k'}}{e^{s_k}} = e^{s_k' - s_k},\qquad
\mathrm{NTE}_k^j = e^{s_k' - s_k} - 1,\qquad
\mathrm{NTE}_k = \sum_{j=1}^{J}\mathrm{NTE}_k^j. \tag{Eq.9-11}$$

被模型判为目标类的样本中，`argmax_k NTE_k` 被认为是它的原始标签（Eq.12）；
先用 t-SNE 降维再用 Birch 聚类，把“目标类的干净图”和“中毒图”分开，重标后者即可去除后门，无需重训练。

## 判定语义

| 判定 | 含义 |
|---|---|
| `CONFIRMED` | 所有核心子检查在容差内通过。只有 Tier A 能获得此判定。 |
| `PARTIAL` | 部分通过；或缩比实验方向一致但规模不同。**`scope: proxy` 永不判 CONFIRMED。** |
| `CONTRADICTED` | 可直接核对的数值超差，或普适断言存在反例。 |
| `NOT_REPRODUCIBLE` | 需要容器外资源（Tier C）。 |
| `ERROR` | 运行异常。正常交付物中不允许出现。 |

## 三层验证

- **Tier A（精确层）**：附录算例、公式一致性、表格算术、文字断言、证明步骤的代数检查、官方代码静态审计。
- **Tier B（缩比实证层）**：CIFAR-10 子集 + SmallCNN + BadNets/Blend，检验方法机制是否按论文方向工作。
  所有结论明确标注为 proxy，不与论文表格逐位比较。
- **Tier C（不可复现层）**：ResNet18/ImageNet/语音文本/13 组自适应训练等，说明原因与所需资源。


In [1]:
import json
import os
import sys
from pathlib import Path

# 定位仓库根目录：容器内是 /app，本地打开时可能是 notebooks/ 或其子目录。
ROOT = Path(os.environ.get("EBBA_ROOT", Path.cwd())).resolve()
if not (ROOT / "src" / "ebba").exists():
    for cand in [Path.cwd(), *Path.cwd().parents]:
        if (cand / "src" / "ebba").exists():
            ROOT = cand
            break
sys.path.insert(0, str(ROOT / "src"))
os.environ.setdefault("EBBA_ROOT", str(ROOT))

import matplotlib.pyplot as plt
import numpy as np

from ebba import config, nbutil
from ebba.claims import load_registry, run, run_many

plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
print("project root:", ROOT)
print("artifacts   :", config.ARTIFACTS_DIR)
print("reports     :", config.REPORTS_DIR)
print("fast mode   :", config.FAST)


project root: /app
artifacts   : /app/artifacts
reports     : /app/reports
fast mode   : False


## 环境指纹

先记录 Python、NumPy、PyTorch、设备与路径。出现跨机器数值差异时，这张指纹是排查的起点；
`artifacts/manifest.json` 还记录了三个 checkpoint 的 SHA256 与训练指标。


In [2]:
nbutil.env_fingerprint()


{
  "python": "3.11.10",
  "platform": "Linux-6.18.54.1-microsoft-standard-WSL2-x86_64-with-glibc2.36",
  "numpy": "1.26.4",
  "root": "/app",
  "artifacts": "/app/artifacts",
  "reports": "/app/reports",
  "fast": false,
  "device_env": "cpu",
  "torch": "2.7.1+cpu",
  "cuda_available": false
}


{'python': '3.11.10',
 'platform': 'Linux-6.18.54.1-microsoft-standard-WSL2-x86_64-with-glibc2.36',
 'numpy': '1.26.4',
 'root': '/app',
 'artifacts': '/app/artifacts',
 'reports': '/app/reports',
 'fast': False,
 'device_env': 'cpu',
 'torch': '2.7.1+cpu',
 'cuda_available': False}

## 声明索引

`src/ebba/claims/registry.yaml` 是唯一事实源：每条声明登记 ID、层级、范围、论文位置、容差和所属 Notebook。
运行结果写入 `reports/claims/<ID>.json`。


In [3]:
registry = load_registry()
from collections import Counter

print("claim count:", len(registry))
print("by tier   :", dict(sorted(Counter(v["tier"] for v in registry.values()).items())))
print("by scope  :", dict(Counter(v["scope"] for v in registry.values())))
print()
for cid, spec in registry.items():
    print(f'{cid:>4}  T{spec["tier"]}  {spec["scope"]:<6}  {spec["notebook"]:<18}  {spec["title_zh"]}')


claim count: 39
by tier   : {'A': 20, 'B': 14, 'C': 5}
by scope  : {'exact': 25, 'proxy': 14}

 A01  TA  exact   01_formula_audit    附录 B 算例：softmax 与能量数值是否可复算
 A02  TA  exact   01_formula_audit    Eq.3 是“最大值置 0”，而非减去最大值
 A03  TA  exact   01_formula_audit    Eq.5、官方 README 与官方代码的归一化公式一致
 A04  TA  exact   01_formula_audit    Eq.7 的求和下标与方差分母自洽
 A05  TA  exact   01_formula_audit    Eq.9-10 的 TE/NTE 恒等式及均值为 0
 A06  TA  exact   01_formula_audit    Eq.9-10 与 Eq.4 使用同一温度 T
 A07  TA  exact   02_theory           Lemma 2 证明中 Eq.30 的代数等价性
 A08  TA  exact   02_theory           Lemma 2 对每一种损坏都成立
 A09  TA  exact   02_theory           Lemma 1（脏标签）的 NTK 核和占优
 A10  TA  exact   02_theory           Lemma 1（干净标签）的 NTK 核和占优
 A11  TA  exact   03_table_audit      BA 下界 (K-1)/K × ba 对所有 Table 4 行成立
 A12  TA  exact   03_table_audit      所有攻击的 ASR 均训练到 98.8% 以上
 A13  TA  exact   03_table_audit      Table 6 的 GTSRB 测试集大小与标准数据集一致
 A14  TA  exact   03_table_audit      四种聚类方法的 PDR 差异很小（Table 5/7）
 A15  TA  exact   

## 代理实验的产物与门槛

镜像构建阶段会训练三个 SmallCNN（clean / BadNets / Blend），并把数据子集、checkpoint 和指标写入
`artifacts/manifest.json`。构建门槛检验的是**代理实验台是否可用**（模型精度、攻击成功率、干净模型不被误报），
而不是论文结论本身；NTE 恢复率等属于被检验的结果，记录但不设门槛，避免为通过构建而调参。


In [4]:
manifest_path = config.ARTIFACTS_DIR / "manifest.json"
if manifest_path.exists():
    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    print(json.dumps(manifest["metrics"], indent=2, ensure_ascii=False))
    print("\nprotocol:", json.dumps(manifest["protocol"], ensure_ascii=False))
    print("checkpoint sha256:")
    for rel, digest in manifest["sha256"].items():
        if rel.startswith("checkpoints/"):
            print(f"  {rel}: {digest[:16]}...")
else:
    print("未找到 artifacts/manifest.json。请先运行：python scripts/build_artifacts.py --device cpu")


{
  "clean": {
    "ba": 0.807,
    "asr": 0.115
  },
  "badnets": {
    "ba": 0.7905,
    "asr": 0.937
  },
  "blend": {
    "ba": 0.804,
    "asr": 0.984
  }
}

protocol: {"train_per_class": 2000, "test_per_class": 200, "ood_per_class": 50, "epochs": 30, "poison_rate": 0.1, "target": 0, "temperature": 2.0, "lambda": 3.0, "proxy_lambda": 2.2, "robust_augment": true}
checkpoint sha256:
  checkpoints/clean.pt: d4d6a1324d9f799b...
  checkpoints/badnets.pt: 1e44a4739c5553e2...
  checkpoints/blend.pt: 4044dd7bf5231a58...


## 如何运行

```bash
# 容器内（推荐）
docker compose run --rm ebba verify          # 约 4 分钟，产出 reports/report.html
docker compose run --rm ebba test            # pytest
docker compose run --rm ebba retrain --attack badnets

# 本地
python scripts/build_artifacts.py --device cpu
python scripts/run_papermill.py
python scripts/make_report.py
```

Notebook 顺序：`01_formula_audit` → `02_theory` → `03_table_audit` → `04_detection` → `05_removal` → `06_verdicts`。
每个 Notebook 可单独运行，但 `06_verdicts` 需要前五个已经写出结果。

## 重要限制（先读）

1. 缩比实验使用 **CIFAR-100 测试集代理“任务无关互联网图像”**，共享自然图像域，但并不是论文的数据来源。
2. 只训练 **SmallCNN、30 epochs、两种攻击**；不覆盖 ResNet18、200 epochs、WaNet/FIBA/DUBA。
   训练使用了轻度的噪声/模糊/对比度增广，这是为了补偿缩比模型远低于论文 ResNet18 的损坏鲁棒性。
3. 因此 Tier B 的 `PARTIAL` 只表示“机制方向在缩比设定下成立”，不能引用为论文 Table 3/4 的数值复现。
4. `CONTRADICTED` 针对的是**某条可核对断言**，不代表 EBBA 方法整体无效。
